In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import re
import json
import math
import random
from pathlib import Path

import numpy as np
import scipy.ndimage as ndimage
from scipy.interpolate import interp1d
from sklearn.model_selection import KFold
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")


Executing on device: cuda


In [3]:
VAL_DIR = Path("/content/drive/MyDrive/val_brats2020")
MODEL_WEIGHTS_PATH = Path("/content/unet3d_final_fp32.pth")
CALIBRATION_CACHE_PATH = Path("/content/calibration_cache.json")

BASE_FEATURES = (16, 32, 64, 128)
BOTTLENECK_FEATURES = 256
DROPOUT = 0.20

MC_DROPOUT_PASSES = 20
VOXEL_SPACING = (1.0, 1.0, 1.0)
VOXEL_VOLUME_CM3 = float(np.prod(VOXEL_SPACING)) / 1000.0


In [4]:
class ResBlock3D(nn.Module):
    """
    3D Residual Convolutional Block (Milletari et al., 2016; Çiçek et al., 2016)
    """
    def __init__(self, in_ch, out_ch, dropout=0.2):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm3d(out_ch, affine=True)
        self.relu = nn.LeakyReLU(negative_slope=0.01, inplace=True)
        self.drop = nn.Dropout3d(dropout)
        self.conv2 = nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm3d(out_ch, affine=True)

        if in_ch != out_ch:
            self.skip = nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False)
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        out = self.relu(self.norm1(self.conv1(x)))
        out = self.drop(out)
        out = self.norm2(self.conv2(out))
        out = self.relu(out + residual)
        return out

class ResidualUNet3D(nn.Module):
    """
    Baseline 3D Residual U-Net for BraTS
    """
    def __init__(self, in_channels=4, num_classes=4, features=(16, 32, 64, 128), bottleneck=256, dropout=0.2):
        super().__init__()
        f1, f2, f3, f4 = features

        # Encoder
        self.stem = nn.Conv3d(in_channels, f1, kernel_size=3, padding=1, bias=False)
        self.enc1 = ResBlock3D(f1, f1, dropout)
        self.down1 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.enc2 = ResBlock3D(f1, f2, dropout)
        self.down2 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.enc3 = ResBlock3D(f2, f3, dropout)
        self.down3 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.enc4 = ResBlock3D(f3, f4, dropout)
        self.down4 = nn.MaxPool3d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck = ResBlock3D(f4, bottleneck, dropout)

        # Decoder
        self.up4 = nn.ConvTranspose3d(bottleneck, f4, kernel_size=2, stride=2)
        self.dec4 = ResBlock3D(f4 + f4, f4, dropout)

        self.up3 = nn.ConvTranspose3d(f4, f3, kernel_size=2, stride=2)
        self.dec3 = ResBlock3D(f3 + f3, f3, dropout)

        self.up2 = nn.ConvTranspose3d(f3, f2, kernel_size=2, stride=2)
        self.dec2 = ResBlock3D(f2 + f2, f2, dropout)

        self.up1 = nn.ConvTranspose3d(f2, f1, kernel_size=2, stride=2)
        self.dec1 = ResBlock3D(f1 + f1, f1, dropout)

        # Output Segmentation Head
        self.head = nn.Conv3d(f1, num_classes, kernel_size=1)

    def forward(self, x):
        x = self.stem(x)
        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))
        e3 = self.enc3(self.down2(e2))
        e4 = self.enc4(self.down3(e3))

        b = self.bottleneck(self.down4(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.head(d1)


In [5]:
def enable_mc_dropout(model):
    """
    Bật dropout khi suy luận (Inference-time MC Dropout) cho toàn bộ các lớp Dropout3d
    """
    for m in model.modules():
        if m.__class__.__name__.startswith('Dropout'):
            m.train()

model = ResidualUNet3D(
    in_channels=4,
    num_classes=4,
    features=BASE_FEATURES,
    bottleneck=BOTTLENECK_FEATURES,
    dropout=DROPOUT
).to(device)

if MODEL_WEIGHTS_PATH.exists():
    checkpoint = torch.load(MODEL_WEIGHTS_PATH, map_location=device)
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)
    print(f"Đã load thành công trọng số từ: {MODEL_WEIGHTS_PATH}")
else:
    raise FileNotFoundError(f"Không tìm thấy file checkpoint tại: {MODEL_WEIGHTS_PATH}")

model.eval()
model.apply(enable_mc_dropout)


Đã load thành công trọng số từ: /content/unet3d_final_fp32.pth


ResidualUNet3D(
  (stem): Conv3d(4, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
  (enc1): ResBlock3D(
    (conv1): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
    (norm1): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=False)
    (relu): LeakyReLU(negative_slope=0.01, inplace=True)
    (drop): Dropout3d(p=0.2, inplace=False)
    (conv2): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
    (norm2): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=False)
    (skip): Identity()
  )
  (down1): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc2): ResBlock3D(
    (conv1): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
    (norm1): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=False)
    (r

In [6]:
def load_val_case_npy(filepath):
    """
    Tải file .npy từ tập val_brats2020.
    Hỗ trợ cả 2 định dạng phổ biến:
    - Dict/Tuple: {'image': [4, D, H, W], 'seg': [D, H, W]}
    - Array ghép: Mảng kích thước [5, D, H, W] trong đó 4 kênh đầu là image, kênh thứ 5 là seg
    """
    data = np.load(filepath, allow_pickle=True)
    if isinstance(data, np.ndarray) and data.dtype == object and data.shape == ():
        item = data.item()
        image = np.asarray(item["image"], dtype=np.float32)
        seg = np.asarray(item["mask"], dtype=np.uint8)
    elif isinstance(data, dict):
        image = np.asarray(data["image"], dtype=np.float32)
        seg = np.asarray(data["mask"], dtype=np.uint8)
    elif isinstance(data, np.ndarray):
        if data.shape[0] == 5:
            image = data[:4].astype(np.float32)
            seg = data[4].astype(np.uint8)
        else:
            raise ValueError(f"Shape không xác định cho mảng npy: {data.shape}")
    else:
        raise ValueError(f"Không hỗ trợ định dạng dữ liệu trong file {filepath}")

    return image, seg

val_files = sorted(list(VAL_DIR.glob("brats2020__BraTS20_Training_*.npy")))
if len(val_files) == 0:
    val_files = sorted(list(VAL_DIR.glob("*.npy")))
print(f"Tìm thấy tổng cộng {len(val_files)} ca validation trong thư mục {VAL_DIR}")


Tìm thấy tổng cộng 74 ca validation trong thư mục /content/drive/MyDrive/val_brats2020


In [7]:
@torch.no_grad()
def predict_mc_dropout_case(model, image, patch_size=(128, 128, 128), overlap=0.5, T=20):
    model.eval()
    model.apply(enable_mc_dropout)

    C, orig_D, orig_H, orig_W = image.shape
    pD, pH, pW = patch_size

    # 1. Tự động Pad nếu ảnh có chiều bất kỳ nhỏ hơn patch_size (128)
    pad_d = max(0, pD - orig_D)
    pad_h = max(0, pH - orig_H)
    pad_w = max(0, pW - orig_W)

    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        image = np.pad(
            image,
            ((0, 0), (0, pad_d), (0, pad_h), (0, pad_w)),
            mode="constant",
            constant_values=0
        )

    _, D, H, W = image.shape

    # 2. Tính các bước trượt (Sliding window)
    step_d = int(pD * (1 - overlap))
    step_h = int(pH * (1 - overlap))
    step_w = int(pW * (1 - overlap))

    steps_d = list(range(0, max(1, D - pD + 1), step_d))
    if steps_d[-1] + pD < D: steps_d.append(D - pD)

    steps_h = list(range(0, max(1, H - pH + 1), step_h))
    if steps_h[-1] + pH < H: steps_h.append(H - pH)

    steps_w = list(range(0, max(1, W - pW + 1), step_w))
    if steps_w[-1] + pW < W: steps_w.append(W - pW)

    prob_wt_runs = []
    prob_tc_runs = []
    prob_et_runs = []

    # 3. Ghép batch 2 lượt MC Dropout chạy cùng lúc để tăng tốc
    batch_t = 2
    for t_step in range(0, T, batch_t):
        current_b = min(batch_t, T - t_step)
        prob_accum = np.zeros((current_b, 4, D, H, W), dtype=np.float32)
        count_accum = np.zeros((D, H, W), dtype=np.float32)

        for sd in steps_d:
            for sh in steps_h:
                for sw in steps_w:
                    sub_vol = image[:, sd:sd+pD, sh:sh+pH, sw:sw+pW]
                    x = torch.from_numpy(np.asarray(sub_vol, dtype=np.float32)).unsqueeze(0).repeat(current_b, 1, 1, 1, 1).to(device)

                    with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                        logits = model(x)
                        probs = torch.softmax(logits, dim=1).cpu().numpy()

                    prob_accum[:, :, sd:sd+pD, sh:sh+pH, sw:sw+pW] += probs
                    count_accum[sd:sd+pD, sh:sh+pH, sw:sw+pW] += 1.0

        count_accum = np.maximum(count_accum, 1.0)
        prob_accum /= count_accum[None, None, ...]

        # 4. Cắt bỏ phần padding thừa để trả về đúng kích thước gốc
        prob_accum = prob_accum[:, :, :orig_D, :orig_H, :orig_W]

        for b in range(current_b):
            p_et = prob_accum[b, 1]
            p_tc = prob_accum[b, 1] + prob_accum[b, 2]
            p_wt = prob_accum[b, 1] + prob_accum[b, 2] + prob_accum[b, 3]

            prob_wt_runs.append(p_wt.astype(np.float16))
            prob_tc_runs.append(p_tc.astype(np.float16))
            prob_et_runs.append(p_et.astype(np.float16))

    return {
        "WT": np.stack(prob_wt_runs, axis=0), # [20, orig_D, orig_H, orig_W]
        "TC": np.stack(prob_tc_runs, axis=0),
        "ET": np.stack(prob_et_runs, axis=0)
    }

In [8]:
def compute_volumetric_cdf_fast(prob_tensor, voxel_vol_cm3, percentiles_levels=np.linspace(0.01, 0.99, 99)):
    """
    Tính Volumetric CDF bằng Soft Volumetry:
    Cộng dồn xác suất tích lũy của từng voxel dọc theo 99 phân vị
    """
    t_prob = torch.from_numpy(prob_tensor.astype(np.float32)).to(device)
    q_levels = torch.from_numpy(percentiles_levels.astype(np.float32)).to(device)

    valid_mask = (t_prob.max(dim=0).values > 0.01)
    if not valid_mask.any():
        return np.zeros(len(percentiles_levels), dtype=np.float64)

    valid_voxels = t_prob[:, valid_mask] # [20, N_valid]
    q_voxels = torch.quantile(valid_voxels, q_levels, dim=0) # [99, N_valid]

    # Tổng xác suất tích lũy
    vol_cdf = q_voxels.sum(dim=1).cpu().numpy().astype(np.float64) * voxel_vol_cm3

    del t_prob, valid_voxels, q_voxels
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return vol_cdf

def compute_gt_volumes(seg, voxel_vol_cm3):
    seg = np.asarray(seg)

    if seg.ndim == 4 and seg.shape[0] == 3:
        return {
            "WT": float(np.sum(seg[0] == 1) * voxel_vol_cm3),
            "TC": float(np.sum(seg[1] == 1) * voxel_vol_cm3),
            "ET": float(np.sum(seg[2] == 1) * voxel_vol_cm3)
        }

    elif seg.ndim == 3:
        unique_labels = set(np.unique(seg))
        if 4 in unique_labels:
            mask_et = (seg == 4)
            mask_tc = np.isin(seg, [1, 4])
            mask_wt = (seg > 0)
        else:
            mask_et = (seg == 1)
            mask_tc = np.isin(seg, [1, 2])
            mask_wt = np.isin(seg, [1, 2, 3])

        return {
            "WT": float(np.sum(mask_wt) * voxel_vol_cm3),
            "TC": float(np.sum(mask_tc) * voxel_vol_cm3),
            "ET": float(np.sum(mask_et) * voxel_vol_cm3)
        }
    else:
        raise ValueError(f"Shape của mask không hợp lệ: {seg.shape}")


In [9]:
PERCENTILES_LEVELS = np.linspace(0.01, 0.99, 99)

all_cdf_volumes = {"WT": [], "TC": [], "ET": []}
all_gt_volumes = {"WT": [], "TC": [], "ET": []}
case_names = []

for case_file in tqdm(val_files, desc="Processing Val Cases for Calibration"):
    case_name = case_file.stem
    case_names.append(case_name)

    image, seg = load_val_case_npy(case_file)
    gt_vols = compute_gt_volumes(seg, VOXEL_VOLUME_CM3)

    mc_preds = predict_mc_dropout_case(model, image, patch_size=(128, 128, 128), overlap=0.5, T=MC_DROPOUT_PASSES)

    for modality in ["WT", "TC", "ET"]:
        cdf_curve = compute_volumetric_cdf_fast(mc_preds[modality], VOXEL_VOLUME_CM3, PERCENTILES_LEVELS)
        all_cdf_volumes[modality].append(cdf_curve)
        all_gt_volumes[modality].append(gt_vols[modality])

for m in ["WT", "TC", "ET"]:
    all_cdf_volumes[m] = np.array(all_cdf_volumes[m]) # [N_val, 99]
    all_gt_volumes[m] = np.array(all_gt_volumes[m])   # [N_val]

print(f"Đã xử lý xong {len(val_files)} ca validation.")


Processing Val Cases for Calibration:   0%|          | 0/74 [00:00<?, ?it/s]

Đã xử lý xong 74 ca validation.


## VER1

In [11]:
def compute_conformal_scores(cdfs_vols, gt_vols, percentiles_levels=PERCENTILES_LEVELS):
    idx_025 = int(np.argmin(np.abs(percentiles_levels - 0.025)))
    idx_500 = int(np.argmin(np.abs(percentiles_levels - 0.500)))
    idx_975 = int(np.argmin(np.abs(percentiles_levels - 0.975)))

    scores = []
    for cdf_vol, gt_vol in zip(cdfs_vols, gt_vols):
        mono_cdf = np.maximum.accumulate(cdf_vol)
        v_low = mono_cdf[idx_025]
        v_med = mono_cdf[idx_500]
        v_high = mono_cdf[idx_975]

        raw_half_width = max((v_high - v_low) / 2.0, 0.0)
        baseline_half_width = max(raw_half_width, 0.15 * v_med + 0.5)

        residual = abs(gt_vol - v_med)
        s = residual / baseline_half_width
        scores.append(s)

    return np.array(scores, dtype=np.float64)

def fit_conformal_scale(scores_train, target_coverage=0.95):
    N = len(scores_train)
    q_level = min(1.0, np.ceil((N + 1) * target_coverage) / N)
    scale = float(np.quantile(scores_train, min(q_level, 0.95)))
    return float(np.clip(scale, 1.05, 2.50))

def evaluate_conformal_coverage(cdfs_vols, gt_vols, scale_factor, percentiles_levels=PERCENTILES_LEVELS):
    idx_025 = int(np.argmin(np.abs(percentiles_levels - 0.025)))
    idx_500 = int(np.argmin(np.abs(percentiles_levels - 0.500)))
    idx_975 = int(np.argmin(np.abs(percentiles_levels - 0.975)))

    covered = 0
    N = len(gt_vols)
    for cdf_vol, gt_vol in zip(cdfs_vols, gt_vols):
        mono_cdf = np.maximum.accumulate(cdf_vol)
        v_low = mono_cdf[idx_025]
        v_med = mono_cdf[idx_500]
        v_high = mono_cdf[idx_975]

        raw_half_width = max((v_high - v_low) / 2.0, 0.0)
        baseline_half_width = max(raw_half_width, 0.15 * v_med + 0.5)

        ci_low = max(0.0, v_med - scale_factor * baseline_half_width)
        ci_high = v_med + scale_factor * baseline_half_width

        if ci_low <= gt_vol <= ci_high:
            covered += 1

    return float(covered / N)

# Chạy lại 3-Fold Cross-Validation
N_VAL = len(val_files)
kf = KFold(n_splits=3, shuffle=True, random_state=SEED)

final_calibration_factors = {}
folds_details = {"WT": [], "TC": [], "ET": []}

for modality in ["WT", "TC", "ET"]:
    cdfs_m = all_cdf_volumes[modality]
    gts_m = all_gt_volumes[modality]

    fold_idx = 1
    for train_idx, test_idx in kf.split(cdfs_m):
        train_cdfs, train_gts = cdfs_m[train_idx], gts_m[train_idx]
        test_cdfs, test_gts = cdfs_m[test_idx], gts_m[test_idx]

        train_scores = compute_conformal_scores(train_cdfs, train_gts)
        a_fold = fit_conformal_scale(train_scores, target_coverage=0.95)
        cov_95 = evaluate_conformal_coverage(test_cdfs, test_gts, a_fold)

        folds_details[modality].append({
            "fold": fold_idx,
            "train_cases": int(len(train_idx)),
            "val_cases": int(len(test_idx)),
            "slope": round(float(a_fold), 4),
            "coverage_95": round(float(cov_95), 4)
        })
        fold_idx += 1

    all_scores = compute_conformal_scores(cdfs_m, gts_m)
    a_final = fit_conformal_scale(all_scores, target_coverage=0.95)
    final_calibration_factors[f"a_{modality.lower()}"] = round(float(a_final), 4)

print("=" * 60)
print("KẾT QUẢ HIỆU CHUẨN CALIBRATION CHUẨN XÁC:")
print("=" * 60)
print(json.dumps(final_calibration_factors, indent=2))

KẾT QUẢ HIỆU CHUẨN CALIBRATION CHUẨN XÁC:
{
  "a_wt": 2.5,
  "a_tc": 2.5,
  "a_et": 2.5
}


In [12]:
calibration_cache = {
    "calibration_factors": final_calibration_factors,
    "validation_details": {
        "dataset": "val_calibration",
        "total_cases": int(N_VAL),
        "mc_dropout_passes": int(MC_DROPOUT_PASSES),
        "method": "OLS Uniform Theoretical Quantile Matching",
        "folds": folds_details
    }
}

with open(CALIBRATION_CACHE_PATH, "w", encoding="utf-8") as f:
    json.dump(calibration_cache, f, indent=2)

print(f"Đã lưu thành công file cấu hình hiệu chuẩn tại: {CALIBRATION_CACHE_PATH}")
print(json.dumps(calibration_cache, indent=2))


Đã lưu thành công file cấu hình hiệu chuẩn tại: /content/calibration_cache.json
{
  "calibration_factors": {
    "a_wt": 2.5,
    "a_tc": 2.5,
    "a_et": 2.5
  },
  "validation_details": {
    "dataset": "val_calibration",
    "total_cases": 74,
    "mc_dropout_passes": 20,
    "method": "OLS Uniform Theoretical Quantile Matching",
    "folds": {
      "WT": [
        {
          "fold": 1,
          "train_cases": 49,
          "val_cases": 25,
          "slope": 2.5,
          "coverage_95": 0.96
        },
        {
          "fold": 2,
          "train_cases": 49,
          "val_cases": 25,
          "slope": 2.5,
          "coverage_95": 0.92
        },
        {
          "fold": 3,
          "train_cases": 50,
          "val_cases": 24,
          "slope": 2.5,
          "coverage_95": 0.9167
        }
      ],
      "TC": [
        {
          "fold": 1,
          "train_cases": 49,
          "val_cases": 25,
          "slope": 2.5,
          "coverage_95": 0.8
        },
     

## VER2

In [10]:
def compute_conformal_scores(cdfs_vols, gt_vols, modality="WT", percentiles_levels=PERCENTILES_LEVELS):
    idx_025 = int(np.argmin(np.abs(percentiles_levels - 0.025)))
    idx_500 = int(np.argmin(np.abs(percentiles_levels - 0.500)))
    idx_975 = int(np.argmin(np.abs(percentiles_levels - 0.975)))

    # Dung sai tương đối và tuyệt đối phù hợp với từng phân vùng u
    # WT: u to, dung sai 25% + 1.0 cm3
    # TC: lõi u, dung sai 25% + 0.8 cm3
    # ET: u nhỏ ngấm thuốc, dung sai 30% + 0.6 cm3
    modality_cfg = {
        "WT": (0.25, 1.0),
        "TC": (0.25, 0.8),
        "ET": (0.30, 0.6)
    }
    rel_factor, abs_floor = modality_cfg.get(modality, (0.25, 0.8))

    scores = []
    for cdf_vol, gt_vol in zip(cdfs_vols, gt_vols):
        mono_cdf = np.maximum.accumulate(cdf_vol)
        v_low = mono_cdf[idx_025]
        v_med = mono_cdf[idx_500]
        v_high = mono_cdf[idx_975]

        raw_half_width = max((v_high - v_low) / 2.0, 0.0)
        baseline_half_width = max(raw_half_width, rel_factor * v_med + abs_floor)

        residual = abs(gt_vol - v_med)
        s = residual / baseline_half_width
        scores.append(s)

    return np.array(scores, dtype=np.float64)

def fit_conformal_scale(scores_train, target_coverage=0.95):
    N = len(scores_train)
    q_level = min(1.0, np.ceil((N + 1) * target_coverage) / N)
    scale = float(np.quantile(scores_train, min(q_level, 0.95)))
    # Nới trần lên 3.50 để không bị chặn cứng tại 2.50
    return float(np.clip(scale, 1.05, 3.50))

def evaluate_conformal_coverage(cdfs_vols, gt_vols, scale_factor, modality="WT", percentiles_levels=PERCENTILES_LEVELS):
    idx_025 = int(np.argmin(np.abs(percentiles_levels - 0.025)))
    idx_500 = int(np.argmin(np.abs(percentiles_levels - 0.500)))
    idx_975 = int(np.argmin(np.abs(percentiles_levels - 0.975)))

    modality_cfg = {
        "WT": (0.25, 1.0),
        "TC": (0.25, 0.8),
        "ET": (0.30, 0.6)
    }
    rel_factor, abs_floor = modality_cfg.get(modality, (0.25, 0.8))

    covered = 0
    N = len(gt_vols)
    for cdf_vol, gt_vol in zip(cdfs_vols, gt_vols):
        mono_cdf = np.maximum.accumulate(cdf_vol)
        v_low = mono_cdf[idx_025]
        v_med = mono_cdf[idx_500]
        v_high = mono_cdf[idx_975]

        raw_half_width = max((v_high - v_low) / 2.0, 0.0)
        baseline_half_width = max(raw_half_width, rel_factor * v_med + abs_floor)

        ci_low = max(0.0, v_med - scale_factor * baseline_half_width)
        ci_high = v_med + scale_factor * baseline_half_width

        if ci_low <= gt_vol <= ci_high:
            covered += 1

    return float(covered / N)

# Chạy lại 3-Fold Cross-Validation
N_VAL = len(val_files)
kf = KFold(n_splits=3, shuffle=True, random_state=SEED)

final_calibration_factors = {}
folds_details = {"WT": [], "TC": [], "ET": []}

for modality in ["WT", "TC", "ET"]:
    cdfs_m = all_cdf_volumes[modality]
    gts_m = all_gt_volumes[modality]

    fold_idx = 1
    for train_idx, test_idx in kf.split(cdfs_m):
        train_cdfs, train_gts = cdfs_m[train_idx], gts_m[train_idx]
        test_cdfs, test_gts = cdfs_m[test_idx], gts_m[test_idx]

        train_scores = compute_conformal_scores(train_cdfs, train_gts, modality=modality)
        a_fold = fit_conformal_scale(train_scores, target_coverage=0.95)
        cov_95 = evaluate_conformal_coverage(test_cdfs, test_gts, a_fold, modality=modality)

        folds_details[modality].append({
            "fold": fold_idx,
            "train_cases": int(len(train_idx)),
            "val_cases": int(len(test_idx)),
            "slope": round(float(a_fold), 4),
            "coverage_95": round(float(cov_95), 4)
        })
        fold_idx += 1

    all_scores = compute_conformal_scores(cdfs_m, gts_m, modality=modality)
    a_final = fit_conformal_scale(all_scores, target_coverage=0.95)
    final_calibration_factors[f"a_{modality.lower()}"] = round(float(a_final), 4)

print("=" * 60)
print("KẾT QUẢ HIỆU CHUẨN CALIBRATION TỰ NHIÊN:")
print("=" * 60)
print(json.dumps(final_calibration_factors, indent=2))

KẾT QUẢ HIỆU CHUẨN CALIBRATION TỰ NHIÊN:
{
  "a_wt": 1.75,
  "a_tc": 2.867,
  "a_et": 2.6526
}


In [11]:
calibration_cache = {
    "calibration_factors": final_calibration_factors,
    "validation_details": {
        "dataset": "val_calibration",
        "total_cases": int(N_VAL),
        "mc_dropout_passes": int(MC_DROPOUT_PASSES),
        "method": "OLS Uniform Theoretical Quantile Matching",
        "folds": folds_details
    }
}

with open(CALIBRATION_CACHE_PATH, "w", encoding="utf-8") as f:
    json.dump(calibration_cache, f, indent=2)

print(f"Đã lưu thành công file cấu hình hiệu chuẩn tại: {CALIBRATION_CACHE_PATH}")
print(json.dumps(calibration_cache, indent=2))


Đã lưu thành công file cấu hình hiệu chuẩn tại: /content/calibration_cache.json
{
  "calibration_factors": {
    "a_wt": 1.75,
    "a_tc": 2.867,
    "a_et": 2.6526
  },
  "validation_details": {
    "dataset": "val_calibration",
    "total_cases": 74,
    "mc_dropout_passes": 20,
    "method": "OLS Uniform Theoretical Quantile Matching",
    "folds": {
      "WT": [
        {
          "fold": 1,
          "train_cases": 49,
          "val_cases": 25,
          "slope": 1.6815,
          "coverage_95": 0.92
        },
        {
          "fold": 2,
          "train_cases": 49,
          "val_cases": 25,
          "slope": 1.7509,
          "coverage_95": 0.96
        },
        {
          "fold": 3,
          "train_cases": 50,
          "val_cases": 24,
          "slope": 1.6577,
          "coverage_95": 0.9167
        }
      ],
      "TC": [
        {
          "fold": 1,
          "train_cases": 49,
          "val_cases": 25,
          "slope": 2.1275,
          "coverage_95": 0.